# D2 — Default Risk Classifier
**Notebook:** FinSight_LoanDecider_D2_Classifier  
**Dataset:** 44,991 rows | 77.8% default rate  

### Steps
- Feature selection + encoding
- Train/test split (80/20)
- Random Forest (n_estimators=150)
- Metrics: Accuracy, Precision, Recall, F1, ROC-AUC
- Threshold analysis
- Risk tier assignment (High/Medium/Low)
- Default Probability Analysis


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.calibration import CalibratedClassifierCV

df = pd.read_csv('/content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/Data/loan_data_clean.csv')

print(f"Shape: {df.shape}")
print(df['default_flag'].value_counts())
print(f"\nRemaining dataset default rate: {df['default_flag'].mean()*100:.1f}%")
df.columns



Mounted at /content/drive
Shape: (32574, 14)
default_flag
0    25466
1     7108
Name: count, dtype: int64

Remaining dataset default rate: 21.8%


Index(['loan_id', 'person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_status', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length',
       'default_flag'],
      dtype='object')

In [ ]:
# Features and target
features = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
            'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
            'cb_person_default_on_file', 'person_home_ownership',
            'loan_intent', 'loan_grade']

target = 'default_flag'

X = df[features].copy()
y = df[target]

# Encode categoricals
categorical_cols = ['person_home_ownership', 'loan_intent', 'loan_grade','cb_person_default_on_file']
encoders = {}

for col in categorical_cols:
    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col])
    encoders[col] = encoder

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"\nTrain default rate: {y_train.mean()*100:.1f}%")
print(f"Test default rate: {y_test.mean()*100:.1f}%")

Training rows: 26059
Test rows: 6515

Train default rate: 21.8%
Test default rate: 21.8%


In [ ]:
# Model training
# Class imbalance check
default_rate = y_train.mean()
print(f"Default rate in training: {default_rate*100:.1f}%")

# Train Random Forest on the balanced dataset

# SMOTE + Random Forest
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        class_weight='balanced'))
])

pipeline.fit(X_train, y_train)
# Calibrate
model = CalibratedClassifierCV(pipeline, cv=3, method='sigmoid')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nModel trained successfully")

# Predict
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metrics
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"\n=== MODEL PERFORMANCE ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision : {precision_score(y_test, y_pred):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
print(f"F1 Score  : {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob):.3f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

Default rate in training: 21.8%

Model trained successfully
Training rows: 26059
Test rows: 6515

=== MODEL PERFORMANCE ===
Accuracy  : 0.912
Precision : 0.862
Recall    : 0.711
F1 Score  : 0.779
ROC-AUC   : 0.919

Confusion Matrix:
[[4931  162]
 [ 411 1011]]


"I trained a Random Forest on a balanced version of the synthetic dataset. Since the downstream system (Mistral alerts) depends on meaningful probability estimates, I calibrated the model using Platt scaling (CalibratedClassifierCV with sigmoid). The deployed model is therefore the calibrated Random Forest."

"I balanced the synthetic dataset through undersampling, trained a standard Random Forest, and calibrated the output probabilities for downstream risk explanation."

In [ ]:
#threshold analysis:
from sklearn.metrics import recall_score, precision_score

for threshold in [0.5, 0.4, 0.3, 0.2]:
    y_pred_thresh = (y_prob >= threshold).astype(int)
    recall = recall_score(y_test, y_pred_thresh)
    precision = precision_score(y_test, y_pred_thresh)
    print(f"Threshold {threshold} | Recall: {recall:.3f} | Precision: {precision:.3f}")

Threshold 0.5 | Recall: 0.711 | Precision: 0.862
Threshold 0.4 | Recall: 0.737 | Precision: 0.816
Threshold 0.3 | Recall: 0.763 | Precision: 0.750
Threshold 0.2 | Recall: 0.799 | Precision: 0.656


In [ ]:
test_input = pd.DataFrame([{
    'person_age': 35,
    'person_income': 120000,
    'person_emp_length': 10,
    'loan_amnt': 5000,
    'loan_int_rate': 7.5,
    'loan_percent_income': round(5000/120000, 4),
    'cb_person_cred_hist_length': 8,
    'cb_person_default_on_file': 0,
    'person_home_ownership': 'OWN',
    'loan_intent': 'PERSONAL',
    'loan_grade': 'A'
}])

for col in categorical_cols:
    test_input[col] = encoders[col].transform(test_input[col])

prob = float(model.predict_proba(test_input[features])[0][1])
print(f"Low risk test probability: {prob:.3f}")
print(f"Risk tier: {'High' if prob >= 0.6 else 'Medium' if prob >= 0.3 else 'Low'}")

Low risk test probability: 0.023
Risk tier: Low


In [ ]:
test_input_high = pd.DataFrame([{
    'person_age': 22,
    'person_income': 15000,
    'person_emp_length': 0,
    'loan_amnt': 10000,
    'loan_int_rate': 18.5,
    'loan_percent_income': round(10000/15000, 4),
    'cb_person_cred_hist_length': 1,
    'cb_person_default_on_file': 1,
    'person_home_ownership': 'RENT',
    'loan_intent': 'PERSONAL',
    'loan_grade': 'G'
}])

for col in categorical_cols:
    test_input_high[col] = encoders[col].transform(test_input_high[col])

prob_high = float(model.predict_proba(test_input_high[features])[0][1])
print(f"High risk test probability: {prob_high:.3f}")
print(f"Risk tier: {'High' if prob_high >= 0.6 else 'Medium' if prob_high >= 0.3 else 'Low'}")

High risk test probability: 0.922
Risk tier: High


In [ ]:
# Score the test dataset and assign risk tiers and save model,encoder, thershold details

import joblib
import json

base_path = '/content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/D2_Classifier/Models'

# Save
joblib.dump(model, f'{base_path}/calibrated_rf_model.pkl')
joblib.dump(encoders, f'{base_path}/label_encoders.pkl')

thresholds = {"low_threshold": 0.3, "high_threshold": 0.6}
with open(f'{base_path}/risk_thresholds.json', 'w') as f:
    json.dump(thresholds, f)

# Score test set
test_df = X_test.copy()
test_df['default_probability'] = y_prob
test_df['default_flag'] = y_test.values
test_df['loan_id'] = df.iloc[X_test.index]['loan_id'].values

test_df['risk_tier'] = pd.cut(
    test_df['default_probability'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High']
)

print(f"=== RISK TIER DISTRIBUTION ===")
print(test_df['risk_tier'].value_counts())
print(f"\nModel and encoders saved to: {base_path}")
print(f"Thresholds: {thresholds}")

# Save scored test set
test_df.to_csv('/content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/D2_Classifier/finsight_v2_scored.csv', index=False)
print("Scored test set saved")

=== RISK TIER DISTRIBUTION ===
risk_tier
Low       5069
High      1082
Medium     364
Name: count, dtype: int64

Model and encoders saved to: /content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/D2_Classifier/Models
Thresholds: {'low_threshold': 0.3, 'high_threshold': 0.6}
Scored test set saved


In [ ]:
print(test_df['default_probability'].value_counts().head(10))
print(f"\nUnique probability values: {test_df['default_probability'].nunique()}")
print(f"\nProbability distribution:")
print(test_df['default_probability'].describe())

In [ ]:
# Feauture IMportance

base_rf = model.calibrated_classifiers_[0].estimator.named_steps['classifier']
importances = pd.Series(base_rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

loan_percent_income           0.204499
loan_int_rate                 0.190096
person_income                 0.157360
person_emp_length             0.079780
loan_amnt                     0.074303
loan_grade                    0.070461
loan_intent                   0.069813
person_home_ownership         0.068914
person_age                    0.039993
cb_person_cred_hist_length    0.033428
cb_person_default_on_file     0.011352
dtype: float64


## D2 — Model Performance Summary

### Dataset
- Source: Credit Risk Dataset (Kaggle) — 32,581 rows, cleaned to 32,574
- Default rate: 21.8% (realistic for retail lending)
- Train/test split: 80/20 stratified

### Model: Random Forest (n_estimators=150) + SMOTE + Sigmoid Calibration
| Metric | Value |
|---|---|
| Accuracy | 0.912 |
| Precision | 0.862 |
| Recall | 0.711 |
| F1 Score | 0.779 |
| ROC-AUC | 0.919 |
| Threshold | 0.3 |

### Threshold Decision
Threshold 0.3 selected — Recall 0.763, Precision 0.750.
In credit risk, missing a defaulter costs principal loss.
Threshold lowered from 0.5 to 0.3 to catch more defaulters
at acceptable precision.

### Feature Importance — Top 3
1. loan_percent_income (20.4%) — debt burden signal
2. loan_int_rate (19.0%) — risk pricing signal  
3. person_income (15.7%) — repayment capacity

### Class Imbalance
21.8% default rate. SMOTE applied inside